# 01 — Pyvium 入门

**Pyvium** 是对 IviumSoft「软件开发驱动 DLL」的 Python 封装，让你可以通过编程方式控制 Ivium 电化学设备（CompactStat 等）。

### 环境要求

- **仅支持 Windows** — 底层 DLL 为 Windows 专用
- **必须安装 IviumSoft**（4.1239 或兼容版本）
- 调用大多数 Pyvium 函数前，**IviumSoft 必须处于运行状态**
- Python 3.11 或 3.12

---

## 库架构

Pyvium 提供三个独立层：

```
┌─────────────────────────────────────────────────────────┐
│  from pyvium import Pyvium   ← 推荐用于实验             │
│  高级封装 — Pythonic 命名，抛出异常                     │
├─────────────────────────────────────────────────────────┤
│  from pyvium import Core     ← 底层 DLL 调用            │
│  原始 DLL 封装 — 返回 (result_code, value) 元组         │
├─────────────────────────────────────────────────────────┤
│  from pyvium import Tools    ← 无需硬件                 │
│  IDF 文件解析、CSV 导出 — 支持离线使用                  │
└─────────────────────────────────────────────────────────┘
```

本系列笔记本介绍 **Pyvium** 层（推荐 API）。Tools 层请参阅笔记本 `07_data_processing.ipynb`。

## 1. 导入库

In [ ]:
from pyvium import Pyvium

# 导入错误类型，用于显式异常处理
from pyvium.errors import (
    DriverNotOpenError,
    IviumSoftNotRunningError,
    DeviceNotConnectedToIviumSoftError,
    NoDeviceDetectedError,
    DeviceBusyError,
)

print("pyvium 导入成功")

## 2. 版本校验

连接硬件前，请确认 pyvium 捆绑的 DLL 与已安装的 IviumSoft 版本兼容。版本不匹配会导致静默失败或崩溃。

> **注意：** 必须先调用 `open_driver()`，才能调用其他函数。它会建立与 IviumSoft 的通信通道。

In [ ]:
Pyvium.open_driver()

print(f"DLL 版本       : {Pyvium.get_dll_version()}")
print(f"DLL 版本字符串 : {Pyvium.get_dll_version_string()}")
print(f"IviumSoft 版本 : {Pyvium.get_iviumsoft_version()}")
print(f"所需 DLL 版本  : {Pyvium.get_version_host()}")
print(f"版本匹配       : {Pyvium.check_dll_version()}")

## 3. 驱动生命周期

驱动是 Python 与 IviumSoft 之间的通信通道。使用前务必打开，使用完毕后关闭。

| 步骤 | 方法 | 效果 |
|------|--------|-----|
| 1 | `open_driver()` | 将 Python 连接到正在运行的 IviumSoft 进程 |
| 2 | *（执行操作）* | 测量、配置、运行方法…… |
| 3 | `close_driver()` | 释放连接 — IviumSoft 保持运行 |

当驱动已打开时再次调用 `open_driver()` 会自动先关闭再重新打开（可安全多次调用）。

In [ ]:
# 推荐模式：使用 try/finally 确保驱动一定被关闭
try:
    Pyvium.open_driver()
    print("驱动已打开 — 可以与 IviumSoft 通信")

    # ... 在此处放置测量代码 ...

finally:
    Pyvium.close_driver()
    print("驱动已关闭")

## 4. 错误处理

Pyvium 层会将原始 DLL 返回码转换为带类型的 Python 异常。每个异常对应特定的设备状态或命令结果：

| 异常 | DLL 码 | 含义 | 解决方法 |
|-----------|----------|---------|-----|
| `DriverNotOpenError` | — | 未调用 `open_driver()` | 先调用 `open_driver()` |
| `IviumSoftNotRunningError` | — | IviumSoft 进程未运行 | 启动 IviumSoft |
| `DeviceNotConnectedToIviumSoftError` | — | 设备已找到但未在 IviumSoft 中连接 | 在 IviumSoft 中点击连接，或调用 `connect_device()` |
| `NoDeviceDetectedError` | `-1` | 未检测到设备（setter 返回码 -1 时也会抛出） | 检查 USB 线缆和驱动安装 |
| `DeviceBusyError` | — | 设备正在执行测量 | 等待或调用 `abort_method()` |
| `CellOffError` | — | 试图在电池关闭时进行测量 | 先调用 `set_cell_on()` |
| `IllegalCommandError` | `1` | 该设备或当前模式不支持此命令 | 检查连接模式和设备能力 |
| `InvalidStateError` | `3` | 命令有效但设备状态不允许 | 等待当前测量完成 |
| `ValueError`（内置） | `2` | 参数超出范围 — 固件拒绝该值 | 检查参数的有效范围 |

In [ ]:
# 示例：优雅地处理错误
try:
    Pyvium.open_driver()
    Pyvium.connect_device()
    print("设备已连接")

    # ... 测量代码 ...

except IviumSoftNotRunningError:
    print("错误：IviumSoft 未运行，请先启动后重试。")

except DeviceNotConnectedToIviumSoftError:
    print("错误：没有已连接的设备，请检查 USB 线缆和 IviumSoft。")

except NoDeviceDetectedError:
    print("错误：未检测到设备，请检查 USB 驱动。")

except DeviceBusyError:
    print("错误：设备正忙，请等待当前测量完成。")

finally:
    try:
        Pyvium.close_driver()
    except DriverNotOpenError:
        pass  # open_driver() 可能在关闭前已失败

## 5. 检查 IviumSoft 是否运行

在开始测量前，可以在不抛出异常的情况下探测系统状态。

In [ ]:
Pyvium.open_driver()

running = Pyvium.is_iviumsoft_running()
print(f"IviumSoft 运行中: {running}")

if running:
    status_code, status_label = Pyvium.get_device_status()
    print(f"设备状态: {status_code} → '{status_label}'")
    # status_code 含义：
    #  -1 : 无 IviumSoft
    #   0 : 未连接
    #   1 : 可用，空闲
    #   2 : 可用，忙碌（正在运行方法）
    #   3 : 无可用设备

Pyvium.close_driver()

## 6. 最简工作示例

最简单的完整脚本：打开驱动，确认 IviumSoft 正在运行，然后关闭。

In [ ]:
from pyvium import Pyvium
from pyvium.errors import IviumSoftNotRunningError

try:
    Pyvium.open_driver()

    if not Pyvium.is_iviumsoft_running():
        raise IviumSoftNotRunningError()

    status_code, status_label = Pyvium.get_device_status()
    print(f"状态: {status_label}")
    print(f"DLL 正常: {Pyvium.check_dll_version()}")
    print("设置已验证 — 可以继续")

finally:
    try:
        Pyvium.close_driver()
    except Exception:
        pass

---

## 小结

| 概念 | 要点 |
|---------|------------|
| 导入 | `from pyvium import Pyvium` |
| 始终第一步 | `Pyvium.open_driver()` |
| 始终最后一步 | `Pyvium.close_driver()`（使用 `try/finally`） |
| 版本检查 | `Pyvium.check_dll_version()` 应返回 `True` |
| 设备状态 | `Pyvium.get_device_status()` → `(code, label)` |
| 错误类型 | 8 种带类型的异常 — 6 种对应设备/连接状态，2 种对应 setter 返回码 |

## DLL 返回码（Core 层）

如果直接使用 `Core` 层，所有 DLL 函数都返回整数状态码：

| 码 | 含义 |
|------|---------|
| `0` | 成功 |
| `-1` | 无设备 / IviumSoft 未运行 |
| `1` | 非法命令 |
| `2` | 参数超出范围 |
| `3` | 无效状态 |

`Pyvium` 层会自动将这些返回码转换为带类型的异常 — 只有直接使用 `Core` 时才会看到原始码。

## 下一步

- **`02_device_and_instance_management.ipynb`** — 选择设备、管理多个 IviumSoft 实例、按序列号连接
- **`03_direct_mode_basics.ipynb`** — 无需方法文件，直接控制电位/电流
- **`07_data_processing.ipynb`** — 离线解析 IDF 文件（无需硬件）